In [9]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT = "/data/insurance-analytics"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

spark = (
    SparkSession.builder
    .appName("InsuranceAnalytics")
    .master("spark://bd-spark-master:7077")
    .getOrCreate()
)

from src.transformations import (
    clean_customers,
    enrich_claims,
    prepare_products,
    resolve_product_versions
)

print("Spark:", spark.version)
print("Project:", PROJECT)

Spark: 3.3.0
Project: /data/insurance-analytics


In [9]:
import os

PROJECT = "/data/insurance-analytics"
INPUT = "/data/assignment-input"

print("Project exists:", os.path.exists(PROJECT))
print("Project Contents:")
print(os.listdir(PROJECT))
      
print("\nAssignment files:")
print(os.listdir(f"{PROJECT}/data/samples"))

Project exists: True
Project Contents:
['.git', '.gitignore', 'data', 'docs', 'notebooks', 'output', 'README.md', 'src', 'tests']

Assignment files:
['claims.parquet', 'customers.parquet', 'products.parquet']


In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("InsuranceAnalytics-Profiling") \
        .master("spark://bd-spark-master:7077") \
        .getOrCreate()
                 
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Application:", spark.sparkContext.appName)

Spark version: 3.3.0
Spark master: spark://bd-spark-master:7077
Application: InsuranceAnalytics-Profiling


In [6]:
spark.range(10).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [11]:
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Default parallelism: 16


In [14]:
claims = spark.read.parquet(f"{INPUT}/claims.parquet")

customers = spark.read.parquet(f"{INPUT}/customers.parquet")

products = spark.read.parquet(f"{INPUT}/products.parquet")

print("Data loaded successfully")

Data loaded successfully


In [15]:
import os
print(os.listdir(INPUT))

['customers.parquet', 'products.parquet', 'claims.parquet']


In [16]:
print("Claims Schema:")
claims.printSchema()

print("Customers Schema:")
customers.printSchema()

print("Products schema:")
products.printSchema()

Claims Schema:
root
 |-- claim_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- claim_date: string (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- country: string (nullable = true)

Customers Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- lifetime_value: double (nullable = true)

Products schema:
root
 |-- product_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- avg_premium: integer (nullable = true)
 |-- commission_rate: integer (nullable = true)
 |-- risk_category: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- version: integer (nullable = true)

In [17]:
print("Claims:",claims.count())
print("Customers:",customers.count())
print("Products:", products.count())

Claims: 1000000
Customers: 500000
Products: 11


In [ ]:
## 2. Relationship & Business Profiling

In [18]:
products.orderBy("product_code","version").show(50,truncate=False)

+------------+----------------+-----------+---------------+-------------+--------------+----------+----------+-------+
|product_code|product_name    |avg_premium|commission_rate|risk_category|effective_date|end_date  |is_current|version|
+------------+----------------+-----------+---------------+-------------+--------------+----------+----------+-------+
|AUTO        |Auto Insurance  |1000       |8              |MEDIUM       |2020-01-01    |2021-12-31|false     |1      |
|AUTO        |Auto Insurance  |1150       |6              |MEDIUM       |2022-01-01    |2022-12-31|false     |2      |
|AUTO        |Auto Insurance  |1265       |7              |MEDIUM       |2023-01-01    |null      |true      |3      |
|HEALTH      |Health Insurance|2500       |12             |HIGH         |2020-01-01    |2021-12-31|false     |1      |
|HEALTH      |Health Insurance|2875       |10             |HIGH         |2022-01-01    |null      |true      |2      |
|HOME        |Home Insurance  |1200       |6    

In [19]:
products.select(
    "product_code",
    "product_name",
    "risk_category",
    "effective_date",
    "end_date",
    "is_current",
    "version"
).orderBy(
    "product_code",
    "version"
).show(50, truncate=False)

+------------+----------------+-------------+--------------+----------+----------+-------+
|product_code|product_name    |risk_category|effective_date|end_date  |is_current|version|
+------------+----------------+-------------+--------------+----------+----------+-------+
|AUTO        |Auto Insurance  |MEDIUM       |2020-01-01    |2021-12-31|false     |1      |
|AUTO        |Auto Insurance  |MEDIUM       |2022-01-01    |2022-12-31|false     |2      |
|AUTO        |Auto Insurance  |MEDIUM       |2023-01-01    |null      |true      |3      |
|HEALTH      |Health Insurance|HIGH         |2020-01-01    |2021-12-31|false     |1      |
|HEALTH      |Health Insurance|HIGH         |2022-01-01    |null      |true      |2      |
|HOME        |Home Insurance  |LOW          |2020-01-01    |2021-12-31|false     |1      |
|HOME        |Home Insurance  |LOW          |2022-01-01    |2022-12-31|false     |2      |
|HOME        |Home Insurance  |LOW          |2023-01-01    |2023-12-31|false     |3      |

In [22]:
from pyspark.sql import functions as F

claims.groupBy("claim_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100,truncate=False)

+----------+------+
|claim_type|count |
+----------+------+
|AUTO      |250745|
|HOME      |249955|
|LIFE      |249898|
|HEALTH    |249402|
+----------+------+



In [23]:
claims.groupBy("status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50,truncate=False)

+--------+------+
|status  |count |
+--------+------+
|APPROVED|333912|
|REJECTED|333057|
|PENDING |333031|
+--------+------+



In [24]:
customers.groupBy("tier") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50,truncate=False)

+--------+------+
|tier    |count |
+--------+------+
|SILVER  |122955|
|GOLD    |122259|
|PLATINUM|122106|
|BRONZE  |121977|
|null    |10703 |
+--------+------+



In [25]:
claims.groupBy("country") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50,truncate=False)

+-------+------+
|country|count |
+-------+------+
|AT     |724475|
|ES     |25396 |
|DE     |25197 |
|SK     |25183 |
|UK     |25143 |
|NO     |25116 |
|CH     |25032 |
|SE     |24924 |
|FR     |24918 |
|FI     |24903 |
|NL     |24887 |
|IT     |24826 |
+-------+------+



In [27]:
customers.groupBy("country") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50,truncate=False)

+-------+-----+
|country|count|
+-------+-----+
|FR     |42022|
|NO     |41918|
|UK     |41854|
|SK     |41802|
|SE     |41649|
|NL     |41633|
|AT     |41609|
|ES     |41600|
|CH     |41572|
|IT     |41540|
|FI     |41491|
|DE     |41310|
+-------+-----+



In [28]:
claims.select(
    F.min("claim_date").alias("min_claim_date"),
    F.max("claim_date").alias("max_claim_date")
).show()

+--------------+--------------+
|min_claim_date|max_claim_date|
+--------------+--------------+
|    2024-01-01|    2024-12-31|
+--------------+--------------+



In [29]:
products.select(
    "product_code",
    "effective_date",
    "end_date",
    "version",
    "is_current"
).orderBy(
    "product_code",
    "version"
).show(50, truncate=False)

+------------+--------------+----------+-------+----------+
|product_code|effective_date|end_date  |version|is_current|
+------------+--------------+----------+-------+----------+
|AUTO        |2020-01-01    |2021-12-31|1      |false     |
|AUTO        |2022-01-01    |2022-12-31|2      |false     |
|AUTO        |2023-01-01    |null      |3      |true      |
|HEALTH      |2020-01-01    |2021-12-31|1      |false     |
|HEALTH      |2022-01-01    |null      |2      |true      |
|HOME        |2020-01-01    |2021-12-31|1      |false     |
|HOME        |2022-01-01    |2022-12-31|2      |false     |
|HOME        |2023-01-01    |2023-12-31|3      |false     |
|HOME        |2024-01-01    |null      |4      |true      |
|LIFE        |2020-01-01    |2021-12-31|1      |false     |
|LIFE        |2022-01-01    |null      |2      |true      |
+------------+--------------+----------+-------+----------+



In [30]:
# to understand whether every claim has a customer

print("Total claims:", claims.count())

print(
    "Distinct customers in claims:",
    claims.select("customer_id").distinct().count()
)

print(
    "Customers table:",
    customers.select("customer_id").distinct().count()
)

Total claims: 1000000
Distinct customers in claims: 99979
Customers table: 487493


In [31]:
orphan_claims = claims.join(
    customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Claims without matching customer:", orphan_claims.count())

Claims without matching customer: 26024


In [32]:
# Duplicate checks
# Claims
duplicate_claims = (
    claims.groupBy("claim_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate claim IDs:", duplicate_claims.count())

# Customers
duplicate_customers = (
    customers.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", duplicate_customers.count())

# Products
duplicate_products = (
    products.groupBy("product_code", "version")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate product/version records:", duplicate_products.count())

Duplicate claim IDs: 0
Duplicate customer IDs: 7391
Duplicate product/version records: 0


In [33]:
# 10. Check claim amount

claims.select(
    F.min("claim_amount").alias("min_amount"),
    F.max("claim_amount").alias("max_amount"),
    F.avg("claim_amount").alias("avg_amount"),
    F.sum("claim_amount").alias("total_amount")
).show()

+----------+----------+------------------+--------------------+
|min_amount|max_amount|        avg_amount|        total_amount|
+----------+----------+------------------+--------------------+
|    500.02|  49999.98|25251.128111560658|2.525112811156065...|
+----------+----------+------------------+--------------------+



In [34]:
claims.filter(
    F.col("claim_amount") < 0
).count()

0

In [35]:
customer_duplicates = (
    customers
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("record_count"),
        F.min("registration_date").alias("min_registration_date"),
        F.max("registration_date").alias("max_registration_date")
    )
    .filter(F.col("record_count") > 1)
)

customer_duplicates.show(20, truncate=False)

+-----------+------------+---------------------+---------------------+
|customer_id|record_count|min_registration_date|max_registration_date|
+-----------+------------+---------------------+---------------------+
|76         |2           |2020-07-10           |2022-03-14           |
|642        |2           |2023-03-29           |2023-05-13           |
|688        |3           |2020-02-12           |2023-09-25           |
|808        |2           |2021-09-29           |2023-05-19           |
|858        |2           |2021-05-23           |2021-11-28           |
|1025       |2           |2023-01-13           |2023-11-18           |
|1163       |4           |2021-06-09           |2024-10-16           |
|1199       |2           |2020-11-22           |2023-02-25           |
|2488       |4           |2020-12-23           |2024-04-28           |
|2907       |2           |2021-07-16           |2024-04-12           |
|3213       |3           |2020-01-10           |2024-01-28           |
|3566 

In [37]:
# A. Duplicate customer records
duplicate_ids = (
    customer_duplicates
    .select("customer_id")
)

customers.join(
    duplicate_ids,
    on="customer_id",
    how="inner"
).orderBy(
    "customer_id",
    "registration_date"
).show(50, truncate=False)

+-----------+-----------------------+-------+-----------------+--------+--------------+
|customer_id|name                   |country|registration_date|tier    |lifetime_value|
+-----------+-----------------------+-------+-----------------+--------+--------------+
|2          |Customer_2             |SE     |2020-10-25       |PLATINUM|74362.29      |
|2          |Customer_322_duplicate |SE     |2021-07-19       |GOLD    |23137.67      |
|2          |Customer_947_duplicate |SE     |2021-08-04       |PLATINUM|97562.49      |
|2          |Customer_33_duplicate  |SE     |2022-10-24       |BRONZE  |8181.62       |
|2          |Customer_400_duplicate |SE     |2023-07-04       |GOLD    |65335.73      |
|5          |Customer_5437_duplicate|SE     |2021-09-19       |GOLD    |47314.17      |
|5          |Customer_5             |SE     |2022-01-24       |BRONZE  |2616.52       |
|23         |Customer_23            |SE     |2021-02-24       |BRONZE  |2883.08       |
|23         |Customer_204_duplic

In [38]:
# ORphan Claims

orphan_claims.select(
    F.count("*").alias("orphan_claims"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_id")
).show()

+-------------+----------------+
|orphan_claims|null_customer_id|
+-------------+----------------+
|        26024|               0|
+-------------+----------------+



In [39]:
claims.filter(
    F.col("customer_id").isNull()
).count()


0

In [40]:
claims.filter(
    F.col("customer_id").isNotNull()
).join(
    customers.select("customer_id").distinct(),
    on="customer_id",
    how="left_anti"
).count()


26024

In [41]:
def null_profile(df, name):
    print(f"\n===== {name} =====")

    result = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

    result.show(truncate=False)


null_profile(claims, "CLAIMS")
null_profile(customers, "CUSTOMERS")
null_profile(products, "PRODUCTS")


===== CLAIMS =====
+--------+-----------+----------+----------+------------+------+-------+
|claim_id|customer_id|claim_date|claim_type|claim_amount|status|country|
+--------+-----------+----------+----------+------------+------+-------+
|0       |0          |0         |0         |0           |0     |0      |
+--------+-----------+----------+----------+------------+------+-------+


===== CUSTOMERS =====
+-----------+-----+-------+-----------------+-----+--------------+
|customer_id|name |country|registration_date|tier |lifetime_value|
+-----------+-----+-------+-----------------+-----+--------------+
|4137       |10481|0      |13495            |10703|8756          |
+-----------+-----+-------+-----------------+-----+--------------+


===== PRODUCTS =====
+------------+------------+-----------+---------------+-------------+--------------+--------+----------+-------+
|product_code|product_name|avg_premium|commission_rate|risk_category|effective_date|end_date|is_current|version|
+------

In [42]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers_check = customers.withColumn(
    "is_canonical_name",
    F.col("name") == F.concat(
        F.lit("Customer_"),
        F.col("customer_id").cast("string")
    )
)

customers_check.groupBy("is_canonical_name").count().show()

+-----------------+------+
|is_canonical_name| count|
+-----------------+------+
|             null| 14417|
|             true|477212|
|            false|  8371|
+-----------------+------+



In [43]:
duplicate_ids = (
    customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .select("customer_id")
)

customers_check.join(
    duplicate_ids,
    "customer_id",
    "inner"
).groupBy(
    "customer_id"
).agg(
    F.sum(
        F.col("is_canonical_name").cast("int")
    ).alias("canonical_records")
).groupBy(
    "canonical_records"
).count().orderBy("canonical_records").show()

+-----------------+-----+
|canonical_records|count|
+-----------------+-----+
|                1| 7390|
+-----------------+-----+



In [48]:
# Step 2 — Build clean_customers

from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers_prepared = customers.withColumn(
    "is_canonical_name",
    F.col("name") == F.concat(
        F.lit("Customer_"),
        F.col("customer_id").cast("string")
    )
)

customers_prepared.select(
    "customer_id",
    "name",
    "is_canonical_name"
).show(20, truncate=False)

+-----------+-----------+-----------------+
|customer_id|name       |is_canonical_name|
+-----------+-----------+-----------------+
|1          |Customer_1 |true             |
|2          |Customer_2 |true             |
|3          |Customer_3 |true             |
|4          |Customer_4 |true             |
|5          |Customer_5 |true             |
|6          |Customer_6 |true             |
|7          |Customer_7 |true             |
|8          |Customer_8 |true             |
|9          |Customer_9 |true             |
|10         |Customer_10|true             |
|11         |Customer_11|true             |
|12         |Customer_12|true             |
|13         |Customer_13|true             |
|14         |Customer_14|true             |
|15         |Customer_15|true             |
|16         |Customer_16|true             |
|17         |Customer_17|true             |
|18         |Customer_18|true             |
|19         |Customer_19|true             |
|20         |Customer_20|true   

In [49]:
customer_window = Window.partitionBy(
    "customer_id"
).orderBy(
    F.col("is_canonical_name").desc(),
    F.col("registration_date").desc()
)

In [50]:
clean_customers = (
    customers_prepared
    .withColumn(
        "rn",
        F.row_number().over(customer_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "is_canonical_name")
)

In [51]:
print("Original rows:", customers.count())
print("Clean rows:", clean_customers.count())

print(
    "Distinct customer IDs:",
    clean_customers.select("customer_id").distinct().count()
)

Original rows: 500000
Clean rows: 487493
Distinct customer IDs: 487493


In [52]:
print(
    "Duplicate customer IDs after cleaning:",
    clean_customers
        .groupBy("customer_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

Duplicate customer IDs after cleaning: 0


In [56]:
# Validate the Claims → Customer join

customer_lookup = (
    clean_customers
    .select(
        "customer_id",
        "name",
        "country",
        "registration_date",
        "tier",
        "lifetime_value"
    )
    .withColumn("_customer_exists", F.lit(1))
)

claims_enriched = (
    claims
    .join(
        customer_lookup,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "customer_match_status",
        F.when(
            F.col("_customer_exists") == 1,
            "MATCHED"
        ).otherwise("UNMATCHED")
    )
    .drop("_customer_exists")
)


In [57]:
claims_enriched.groupBy(
    "customer_match_status"
).count().show()

+---------------------+------+
|customer_match_status| count|
+---------------------+------+
|              MATCHED|973976|
|            UNMATCHED| 26024|
+---------------------+------+



In [58]:
print("Raw claims:", claims.count())
print("Enriched claims:", claims_enriched.count())

Raw claims: 1000000
Enriched claims: 1000000


In [59]:
raw_customer_ids = customers.select(
    "customer_id"
).distinct()

clean_customer_ids = clean_customers.select(
    "customer_id"
).distinct()

print(
    "Raw customer IDs:",
    raw_customer_ids.count()
)

print(
    "Clean customer IDs:",
    clean_customer_ids.count()
)

print(
    "Customer IDs removed:",
    raw_customer_ids.join(
        clean_customer_ids,
        "customer_id",
        "left_anti"
    ).count()
)

Raw customer IDs: 487493
Clean customer IDs: 487493
Customer IDs removed: 1


In [60]:
print(
    "Customers with NULL customer_id:",
    customers.filter(F.col("customer_id").isNull()).count()
)

Customers with NULL customer_id: 4137


In [61]:
customers_valid = customers.filter(
    F.col("customer_id").isNotNull()
)

customers_prepared = customers_valid.withColumn(
    "is_canonical_name",
    F.col("name") == F.concat(
        F.lit("Customer_"),
        F.col("customer_id").cast("string")
    )
)

customer_window = Window.partitionBy(
    "customer_id"
).orderBy(
    F.col("is_canonical_name").desc(),
    F.col("registration_date").desc()
)

clean_customers = (
    customers_prepared
    .withColumn(
        "rn",
        F.row_number().over(customer_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "is_canonical_name")
)

In [62]:
print("Raw customer rows:", customers.count())

print(
    "Invalid NULL customer IDs:",
    customers.filter(F.col("customer_id").isNull()).count()
)

print("Clean customer rows:", clean_customers.count())

print(
    "Clean distinct customer IDs:",
    clean_customers.select("customer_id").distinct().count()
)

print(
    "NULL IDs in clean customers:",
    clean_customers.filter(
        F.col("customer_id").isNull()
    ).count()
)

Raw customer rows: 500000
Invalid NULL customer IDs: 4137
Clean customer rows: 487492
Clean distinct customer IDs: 487492
NULL IDs in clean customers: 0


In [63]:
raw_customer_ids = (
    customers
    .filter(F.col("customer_id").isNotNull())
    .select("customer_id")
    .distinct()
)

clean_customer_ids = (
    clean_customers
    .select("customer_id")
    .distinct()
)

print(
    "Valid raw customer IDs:",
    raw_customer_ids.count()
)

print(
    "Clean customer IDs:",
    clean_customer_ids.count()
)

print(
    "Valid customer IDs lost:",
    raw_customer_ids.join(
        clean_customer_ids,
        "customer_id",
        "left_anti"
    ).count()
)

Valid raw customer IDs: 487492
Clean customer IDs: 487492
Valid customer IDs lost: 0


In [64]:
# Step 1 — Prepare Products
from pyspark.sql import functions as F

products_prepared = (
    products
    .withColumn(
        "effective_date",
        F.to_date("effective_date")
    )
    .withColumn(
        "end_date",
        F.to_date("end_date")
    )
)

products_prepared.printSchema()

root
 |-- product_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- avg_premium: integer (nullable = true)
 |-- commission_rate: integer (nullable = true)
 |-- risk_category: string (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- version: integer (nullable = true)



In [65]:
# Step 2 — Inspect the prepared product dimension

products_prepared.orderBy(
    "product_code",
    "version"
).show(
    50,
    truncate=False
)

+------------+----------------+-----------+---------------+-------------+--------------+----------+----------+-------+
|product_code|product_name    |avg_premium|commission_rate|risk_category|effective_date|end_date  |is_current|version|
+------------+----------------+-----------+---------------+-------------+--------------+----------+----------+-------+
|AUTO        |Auto Insurance  |1000       |8              |MEDIUM       |2020-01-01    |2021-12-31|false     |1      |
|AUTO        |Auto Insurance  |1150       |6              |MEDIUM       |2022-01-01    |2022-12-31|false     |2      |
|AUTO        |Auto Insurance  |1265       |7              |MEDIUM       |2023-01-01    |null      |true      |3      |
|HEALTH      |Health Insurance|2500       |12             |HIGH         |2020-01-01    |2021-12-31|false     |1      |
|HEALTH      |Health Insurance|2875       |10             |HIGH         |2022-01-01    |null      |true      |2      |
|HOME        |Home Insurance  |1200       |6    

In [66]:
 # Step 3 — Perform the historical join
product_condition = (
    (F.col("claim_type") == F.col("product_code"))
    &
    (F.to_date(F.col("claim_date")) >= F.col("effective_date"))
    &
    (
        F.col("end_date").isNull()
        |
        (F.to_date(F.col("claim_date")) <= F.col("end_date"))
    )
)

In [67]:
claims_with_product = (
    claims_enriched.alias("c")
    .join(
        products_prepared.alias("p"),
        on=product_condition,
        how="left"
    )
)

In [69]:
# 4A. Start from the original claims
claims_enriched = (
    claims.alias("c")
    .join(
        clean_customers.alias("cu"),
        on=F.col("c.customer_id") == F.col("cu.customer_id"),
        how="left"
    )
    .select(
        F.col("c.claim_id"),
        F.col("c.customer_id"),
        F.col("c.claim_date"),
        F.col("c.claim_type"),
        F.col("c.claim_amount"),
        F.col("c.status"),
        F.col("c.country").alias("claim_country"),

        F.col("cu.name").alias("customer_name"),
        F.col("cu.country").alias("customer_country"),
        F.col("cu.registration_date").alias("registration_date"),
        F.col("cu.tier").alias("tier"),
        F.col("cu.lifetime_value").alias("lifetime_value"),

        F.when(
            F.col("cu.customer_id").isNotNull(),
            F.lit("MATCHED")
        ).otherwise(
            F.lit("UNMATCHED")
        ).alias("customer_match_status")
    )
)

In [71]:
claims_enriched.printSchema()

root
 |-- claim_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- claim_date: string (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- claim_country: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- lifetime_value: double (nullable = true)
 |-- customer_match_status: string (nullable = false)



In [72]:
# Step 5 — Validate customer enrichment again
claims_enriched.groupBy(
    "customer_match_status"
).count().show()

+---------------------+------+
|customer_match_status| count|
+---------------------+------+
|              MATCHED|973976|
|            UNMATCHED| 26024|
+---------------------+------+



In [73]:
print("Raw claims:", claims.count())
print("Enriched claims:", claims_enriched.count())

Raw claims: 1000000
Enriched claims: 1000000


In [74]:
# Step 6 Prepare Products
products_prepared = (
    products
    .withColumn(
        "effective_date",
        F.to_date("effective_date")
    )
    .withColumn(
        "end_date",
        F.to_date("end_date")
    )
)

In [75]:
products_prepared.printSchema()

root
 |-- product_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- avg_premium: integer (nullable = true)
 |-- commission_rate: integer (nullable = true)
 |-- risk_category: string (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- version: integer (nullable = true)



In [76]:
# Step 7 — Historical product condition
product_condition = (
    (F.col("c.claim_type") == F.col("p.product_code"))
    &
    (F.to_date(F.col("c.claim_date")) >= F.col("p.effective_date"))
    &
    (
        F.col("p.end_date").isNull()
        |
        (F.to_date(F.col("c.claim_date")) <= F.col("p.end_date"))
    )
)

In [77]:
# Step 7 — Historical product condition
claims_with_product = (
    claims_enriched.alias("c")
    .join(
        products_prepared.alias("p"),
        on=product_condition,
        how="left"
    )
)

In [78]:
# Final Fact Table
fact_claims = claims_with_product.select(
    F.col("c.claim_id"),
    F.col("c.customer_id"),
    F.col("c.claim_date"),
    F.col("c.claim_type"),
    F.col("c.claim_amount"),
    F.col("c.status"),
    F.col("c.claim_country"),

    F.col("c.customer_name"),
    F.col("c.customer_country"),
    F.col("c.registration_date"),
    F.col("c.tier"),
    F.col("c.lifetime_value"),
    F.col("c.customer_match_status"),

    F.col("p.product_code"),
    F.col("p.product_name"),
    F.col("p.avg_premium"),
    F.col("p.commission_rate"),
    F.col("p.risk_category"),
    F.col("p.effective_date").alias("product_effective_date"),
    F.col("p.end_date").alias("product_end_date"),
    F.col("p.version").alias("product_version"),
    F.col("p.is_current").alias("product_is_current")
)

In [79]:
# Step 9 - First Critical TEst
print("Raw claims:", claims.count())
print("Fact claims:", fact_claims.count())

Raw claims: 1000000
Fact claims: 1000000


In [80]:
fact_claims.select(
    F.count("*").alias("total_claims"),
    F.sum(
        F.col("product_code").isNull().cast("int")
    ).alias("unmatched_products")
).show()

+------------+------------------+
|total_claims|unmatched_products|
+------------+------------------+
|     1000000|                 0|
+------------+------------------+



In [81]:
fact_claims.groupBy(
    "claim_type",
    "product_version"
).count().orderBy(
    "claim_type",
    "product_version"
).show()

+----------+---------------+------+
|claim_type|product_version| count|
+----------+---------------+------+
|      AUTO|              3|250745|
|    HEALTH|              2|249402|
|      HOME|              4|249955|
|      LIFE|              2|249898|
+----------+---------------+------+



In [82]:
# Step 1 — Do the final SCD integrity checks
# Check 1 - Multiple product matches
product_match_counts = (
    claims.alias("c")
    .join(
        products_prepared.alias("p"),
        on=product_condition,
        how="left"
    )
    .groupBy("c.claim_id")
    .count()
)

product_match_counts.groupBy("count").count().show()

+-----+-------+
|count|  count|
+-----+-------+
|    1|1000000|
+-----+-------+



In [83]:
print(
    "Claims with multiple product matches:",
    product_match_counts
        .filter(F.col("count") > 1)
        .count()
)

Claims with multiple product matches: 0


In [85]:
# Check 2 — Duplicate claim IDs
print(
    "Duplicate claim IDs in fact:",
    fact_claims
        .groupBy("claim_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

Duplicate claim IDs in fact: 0


In [86]:
# Check 3 — Customer match status
fact_claims.groupBy(
    "customer_match_status"
).count().show()

+---------------------+------+
|customer_match_status| count|
+---------------------+------+
|              MATCHED|973976|
|            UNMATCHED| 26024|
+---------------------+------+



## 3. Production Transformation Validation

In [1]:
import sys

PROJECT = "/data/insurance-analytics"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Project path:", PROJECT)

Project path: /data/insurance-analytics


In [2]:
from src.transformations import (
    clean_customers,
    enrich_claims,
    prepare_products,
    resolve_product_versions
)

print("Production transformations imported successfully")

Production transformations imported successfully


In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("InsuranceAnalytics-Profiling") \
        .master("spark://bd-spark-master:7077") \
        .getOrCreate()
                 
print("Spark version:", spark.version)


Spark version: 3.3.0


In [1]:
print("Kernel working")

Kernel working


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("InsuranceAnalytics")
    .master("spark://bd-spark-master:7077")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.3.0


In [3]:
CLAIMS_PATH = "/data/insurance-analytics/data/samples/claims.parquet"
CUSTOMERS_PATH = "/data/insurance-analytics/data/samples/customers.parquet"
PRODUCTS_PATH = "/data/insurance-analytics/data/samples/products.parquet"

claims = spark.read.parquet(CLAIMS_PATH)
customers = spark.read.parquet(CUSTOMERS_PATH)
products = spark.read.parquet(PRODUCTS_PATH)

print("Claims:", claims.count())
print("Customers:", customers.count())
print("Products:", products.count())

Claims: 1000000
Customers: 500000
Products: 11


In [4]:
import sys

PROJECT = "/data/insurance-analytics"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from src.transformations import (
    clean_customers,
    enrich_claims,
    prepare_products,
    resolve_product_versions
)

print("Production transformations imported successfully")

Production transformations imported successfully


In [5]:
clean_customers_prod = clean_customers(customers)

claims_enriched_prod = enrich_claims(
    claims,
    clean_customers_prod
)

products_prod = prepare_products(products)

fact_claims_prod = resolve_product_versions(
    claims_enriched_prod,
    products_prod
)

In [6]:
print("Clean customers:", clean_customers_prod.count())
print("Enriched claims:", claims_enriched_prod.count())
print("Fact claims:", fact_claims_prod.count())

Clean customers: 487492
Enriched claims: 1000000
Fact claims: 1000000


In [8]:
from pyspark.sql import functions as F
fact_claims_prod.select(
    F.count("*").alias("total_claims"),
    F.sum(
        F.col("product_code").isNull().cast("int")
    ).alias("unmatched_products"),
    F.sum(
        (F.col("customer_match_status") == "UNMATCHED").cast("int")
    ).alias("unmatched_customers")
).show()

+------------+------------------+-------------------+
|total_claims|unmatched_products|unmatched_customers|
+------------+------------------+-------------------+
|     1000000|                 0|              26024|
+------------+------------------+-------------------+



In [10]:
from src.data_quality import (
    profile_claims,
    profile_customers,
    profile_products,
    profile_fact_claims
)

In [11]:
claims_dq = profile_claims(claims)
customers_dq = profile_customers(customers)
products_dq = profile_products(products)
fact_dq = profile_fact_claims(
    fact_claims_prod,
    claims.count()
)

print("CLAIMS")
print(claims_dq)

print("\nCUSTOMERS")
print(customers_dq)

print("\nPRODUCTS")
print(products_dq)

print("\nFACT CLAIMS")
print(fact_dq)

CLAIMS
{'row_count': 1000000, 'duplicate_claim_ids': 0, 'negative_claim_amounts': 0, 'invalid_status': 0, 'invalid_claim_type': 0}

CUSTOMERS
{'row_count': 500000, 'duplicate_customer_ids': 7391, 'null_customer_ids': 4137, 'invalid_tier': 0}

PRODUCTS
{'row_count': 11, 'duplicate_product_versions': 0, 'invalid_date_ranges': 0, 'products_with_invalid_current_version_count': 0}

FACT CLAIMS
{'fact_row_count': 1000000, 'raw_claim_count': 1000000, 'duplicate_claim_ids': 0, 'unmatched_products': 0, 'unmatched_customers': 26024}


In [12]:
from src.analytics import build_analytics_marts

print("Analytics module imported successfully")

Analytics module imported successfully


In [13]:
analytics = build_analytics_marts(
    fact_claims_prod
)

In [14]:
print("Analytics marts created:")
for name in analytics:
    print("-", name)

Analytics marts created:
- kpi_summary
- product_summary
- risk_summary
- customer_tier_summary
- country_summary
- status_summary
- monthly_summary
- customer_summary


In [15]:
analytics["kpi_summary"].show(
    truncate=False
)

+------------+---------------------+--------------------+---------------+---------------+--------------+-------------+--------------+------------+
|total_claims|total_claim_amount   |average_claim_amount|approved_claims|rejected_claims|pending_claims|approval_rate|rejection_rate|pending_rate|
+------------+---------------------+--------------------+---------------+---------------+--------------+-------------+--------------+------------+
|1000000     |2.5251128111560658E10|25251.128111560658  |333912         |333057         |333031        |33.39        |33.31         |33.3        |
+------------+---------------------+--------------------+---------------+---------------+--------------+-------------+--------------+------------+



In [16]:
analytics["product_summary"].show(
    truncate=False
)

+------------+----------------+-------------+-----------+-------------------+--------------------+---------------+---------------+--------------------+-------------+
|product_code|product_name    |risk_category|claim_count|total_claim_amount |average_claim_amount|approved_claims|rejected_claims|estimated_commission|approval_rate|
+------------+----------------+-------------+-----------+-------------------+--------------------+---------------+---------------+--------------------+-------------+
|AUTO        |Auto Insurance  |MEDIUM       |250745     |6.337764608049921E9|25275.736736724244  |83863          |83284          |4.4364352256350875E8|33.45        |
|LIFE        |Life Insurance  |LOW          |249898     |6.31924323182003E9 |25287.29014165792   |83480          |83281          |8.215016201366128E8 |33.41        |
|HOME        |Home Insurance  |LOW          |249955     |6.303804865080116E9|25219.759016943513  |83330          |83464          |3.7822829190479684E8|33.34        |
|HEA

In [17]:
analytics["monthly_summary"].show(
    20,
    truncate=False
)

+-----------+-----------+--------------------+--------------------+---------------+-------------+
|claim_month|claim_count|total_claim_amount  |average_claim_amount|approved_claims|approval_rate|
+-----------+-----------+--------------------+--------------------+---------------+-------------+
|2024-01    |84326      |2.1352809137399993E9|25321.738416858374  |28189          |33.43        |
|2024-02    |79460      |2.004074055009994E9 |25221.168575509615  |26574          |33.44        |
|2024-03    |84862      |2.1424102984599762E9|25245.814362847635  |28276          |33.32        |
|2024-04    |82137      |2.0753648483099806E9|25267.112851820504  |27444          |33.41        |
|2024-05    |84319      |2.1285382328599968E9|25243.874249694574  |28181          |33.42        |
|2024-06    |81694      |2.062216951080015E9 |25243.18739540254   |27290          |33.41        |
|2024-07    |84753      |2.1454451415200076E9|25314.090846577794  |28241          |33.32        |
|2024-08    |84528  

In [18]:
analytics["country_summary"].show(
    20,
    truncate=False
)

+-------------+-----------+---------------------+--------------------+---------------+---------------+-------------+
|claim_country|claim_count|total_claim_amount   |average_claim_amount|approved_claims|rejected_claims|approval_rate|
+-------------+-----------+---------------------+--------------------+---------------+---------------+-------------+
|AT           |724475     |1.8298228009320484E10|25257.224899852285  |241926         |241065         |33.39        |
|ES           |25396      |6.405597909900011E8  |25222.861513230473  |8431           |8585           |33.2         |
|SK           |25183      |6.381064654799991E8  |25338.778758686378  |8386           |8429           |33.3         |
|DE           |25197      |6.358508482399931E8  |25235.180705639286  |8354           |8410           |33.15        |
|NO           |25116      |6.331417421899976E8  |25208.701313505237  |8446           |8254           |33.63        |
|FR           |24918      |6.325047460300003E8  |25383.447549161

In [19]:
analytics["customer_tier_summary"].show(
    truncate=False
)

+--------+-----------+--------------------+--------------------+-------------------------------+----------------+-------------------+
|tier    |claim_count|total_claim_amount  |average_claim_amount|average_customer_lifetime_value|unique_customers|claims_per_customer|
+--------+-----------+--------------------+--------------------+-------------------------------+----------------+-------------------+
|SILVER  |240585     |6.074770133749999E9 |25249.995360267676  |64239.77600883303              |23843           |10.09              |
|BRONZE  |237575     |6.008920760489999E9 |25292.731813069553  |61950.29792601578              |23844           |9.96               |
|PLATINUM|237822     |6.005291188979999E9 |25251.201272296083  |63015.489983453466             |23838           |9.98               |
|GOLD    |236861     |5.971321343500002E9 |25210.2344560734    |60214.01915323823              |23779           |9.96               |
|null    |47157      |1.1908246848400002E9|25252.34185465573  